### IBGE População

Trata o Parquet produzido por baixar_populacao_ibge_2020_2024.py.
Executar no notebook: %run tratar_populacao_ibge_silver.py
Dependencias: pandas e pyarrow. Nao baixa dados nem executa SQL.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import uuid
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
PROJETO = Path(r'D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil')
BRONZE = PROJETO / 'Camada_Bronze'
SILVER = PROJETO / 'Camada_Silver'
ENTRADA = None  # Se houver copias, informe o caminho completo do Parquet escolhido.
ANOS = set(range(2020, 2025))
UF = dict(zip('11 12 13 14 15 16 17 21 22 23 24 25 26 27 28 29 31 32 33 35 41 42 43 50 51 52 53'.split(),
              'RO AC AM RR PA AP TO MA PI CE RN PB PE AL SE BA MG ES RJ SP PR SC RS MS MT GO DF'.split()))
TEXTOS = ['codigo_ibge_7', 'codigo_ibge_6', 'codigo_uf', 'uf', 'municipio',
          'valor_original', 'status_populacao', 'tipo_fonte', 'tabela_fonte', 'url_fonte']
OBRIGATORIAS = TEXTOS + ['ano_analise', 'populacao', 'ano_referencia_populacao',
                         'data_referencia_populacao', 'exige_cautela_temporal']

In [3]:
def encontrar():
    if ENTRADA is not None:
        p = Path(ENTRADA)
        if not p.is_file():
            raise FileNotFoundError(p)
        return p
    if not BRONZE.is_dir():
        raise FileNotFoundError(f'Bronze nao encontrada: {BRONZE}')
    arquivos = sorted(BRONZE.rglob('populacao_municipios_2020_2024.parquet'))
    if len(arquivos) != 1:
        raise ValueError(f'Esperava um arquivo IBGE; encontrei {len(arquivos)}. '
                         f'Ajuste ENTRADA para escolher uma coleta. Arquivos: {arquivos}')
    return arquivos[0]


def tratar(original):
    faltam = set(OBRIGATORIAS) - set(original.columns)
    if faltam:
        raise ValueError(f'Layout diferente do extrator: faltam {sorted(faltam)}')
    df = original[OBRIGATORIAS].copy()
    motivos = pd.Series('', index=df.index, dtype='string')

    def marcar(mask, motivo):
        mask = mask.fillna(False)
        motivos.loc[mask] += motivo + ';'

    for c in TEXTOS:
        df[c] = df[c].astype('string').str.strip().replace('', pd.NA)
    for c in ['ano_analise', 'ano_referencia_populacao', 'populacao']:
        texto = original[c].astype('string').str.strip().replace('', pd.NA)
        n = pd.to_numeric(texto, errors='coerce')
        valido = n.notna() & n.gt(0) & n.le(2147483647) & n.mod(1).eq(0)
        marcar((texto.notna() & ~valido) | (texto.isna() & (c != 'populacao')), c + '_invalido')
        df[c] = n.where(valido).astype('Int64')
    marcar(~df.ano_analise.isin(ANOS), 'ano_fora_recorte')
    cod = df.codigo_ibge_7
    marcar(~cod.str.fullmatch(r'[0-9]{7}', na=False) | ~cod.str[:2].isin(UF), 'codigo_ibge_7_invalido')
    marcar(df.codigo_ibge_6.isna() | df.codigo_ibge_6.ne(cod.str[:6]), 'codigo_ibge_6_divergente')
    marcar(df.codigo_uf.isna() | df.codigo_uf.ne(cod.str[:2]), 'codigo_uf_divergente')
    df['uf'] = df.uf.str.upper()
    marcar(df.uf.isna() | df.uf.ne(cod.str[:2].map(UF)), 'uf_divergente')
    for c in ['municipio', 'tipo_fonte', 'tabela_fonte', 'url_fonte']:
        marcar(df[c].isna(), c + '_ausente')
    ref = df.ano_analise.where(df.ano_analise.ne(2023), 2022)
    marcar(df.ano_referencia_populacao.ne(ref), 'ano_referencia_divergente')
    datas = pd.to_datetime(original.data_referencia_populacao, format='%Y-%m-%d', errors='coerce')
    esperada = ref.astype('string') + df.ano_analise.map({2020:'-07-01',2021:'-07-01',2022:'-08-01',2023:'-08-01',2024:'-07-01'}).astype('string')
    marcar(datas.isna() | datas.ne(pd.to_datetime(esperada, errors='coerce')), 'data_referencia_divergente')
    df['data_referencia_populacao'] = datas.dt.date
    cautela = original.exige_cautela_temporal.astype('string').str.strip().str.lower().map({'true':True,'false':False,'1':True,'0':False}).astype('boolean')
    marcar(cautela.isna() | cautela.ne(df.ano_analise.eq(2023)), 'cautela_temporal_divergente')
    df['exige_cautela_temporal'] = cautela
    status = pd.Series('disponivel', index=df.index, dtype='string').where(df.populacao.notna(), 'indisponivel_na_fonte')
    marcar(df.status_populacao.isna() | df.status_populacao.ne(status), 'status_populacao_divergente')
    df['fl_populacao_ausente'] = df.populacao.isna().astype('boolean')
    ruins = original.loc[motivos.ne('')].copy()
    for c in ruins:
        ruins[c] = ruins[c].astype('string')
    ruins['motivos_rejeicao'] = motivos.loc[ruins.index].str.rstrip(';')
    return df.loc[motivos.eq('')].copy(), ruins


def main():
    entrada = encontrar()
    print(f'Lendo: {entrada}', flush=True)
    original = pd.read_parquet(entrada)
    df, ruins = tratar(original)
    run = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
    dirs = {k: SILVER / k / 'populacao_ibge' / run for k in ('dados','qualidade','documentacao')}
    for p in dirs.values():
        p.mkdir(parents=True, exist_ok=False)
    df['arquivo_origem'] = str(entrada)
    df['tratado_em_utc'] = datetime.now(timezone.utc).isoformat()
    campos = []
    for c in df.columns:
        t = (pa.int64() if c == 'populacao' else pa.int16() if c in ('ano_analise','ano_referencia_populacao')
             else pa.bool_() if c in ('exige_cautela_temporal','fl_populacao_ausente')
             else pa.date32() if c == 'data_referencia_populacao' else pa.string())
        campos.append(pa.field(c, t))
    esquema = pa.schema(campos)
    df = df.sort_values(['ano_analise','codigo_ibge_7']).reset_index(drop=True)
    destino = dirs['dados'] / 'populacao_municipios_2020_2024.parquet'
    tmp = destino.with_suffix('.parquet.part')
    pq.write_table(pa.Table.from_pandas(df, schema=esquema, preserve_index=False), tmp, compression='snappy')
    assert pq.read_metadata(tmp).num_rows == len(df)
    assert len(df) + len(ruins) == len(original)
    tmp.replace(destino)
    ruins.to_parquet(dirs['qualidade'] / 'registros_rejeitados.parquet', index=False)
    duplicadas = df.duplicated(['ano_analise','codigo_ibge_7'], keep=False)
    colisoes = df.duplicated(['ano_analise','codigo_ibge_6'], keep=False)
    df.loc[duplicadas | colisoes].to_csv(dirs['qualidade'] / 'chaves_duplicadas.csv', index=False, sep=';', encoding='utf-8-sig')
    resumo = df.groupby('ano_analise').agg(registros=('codigo_ibge_7','size'), populacoes_disponiveis=('populacao','count'))
    resumo.to_csv(dirs['qualidade'] / 'contagens_por_ano.csv', sep=';', encoding='utf-8-sig')
    ausentes = sorted(ANOS - set(df.ano_analise.dropna().astype(int)))
    baixa = [int(a) for a,n in resumo.registros.items() if n < 5000]
    ufs_ausentes = {str(a): sorted(set(UF.values()) - set(df.loc[df.ano_analise.eq(a),'uf'])) for a in sorted(ANOS)}
    revisao = bool(df.empty or len(ruins) or duplicadas.any() or colisoes.any() or ausentes or baixa or any(ufs_ausentes.values()))
    status = 'REVISAR_ANTES_DA_CARGA' if revisao else 'VALIDADO_COM_RESSALVAS'
    relatorio = {'status':status, 'registros_entrada':len(original), 'registros_silver':len(df),
        'registros_quarentena':len(ruins), 'registros_em_chaves_duplicadas_7':int(duplicadas.sum()),
        'registros_em_chaves_duplicadas_6':int(colisoes.sum()), 'anos_ausentes':ausentes,
        'anos_com_menos_5000_registros':baixa, 'ufs_ausentes_por_ano':ufs_ausentes,
        'populacoes_nulas':int(df.populacao.isna().sum()),
        'motivos_rejeicao':ruins.motivos_rejeicao.str.split(';').explode().value_counts().to_dict(),
        'ressalvas':['2023 utiliza referencia populacional de 2022, conforme metadados do extrator.',
                    'Serie mistura censo e estimativas; diferencas nao equivalem diretamente a crescimento.',
                    'Populacoes ausentes continuam NULL; nao usar como denominador.',
                    'Cobertura verifica anos, UFs e piso de registros; nao certifica lista completa de municipios.'],
        'colunas_extras_bronze_nao_exportadas':sorted(set(original.columns)-set(OBRIGATORIAS)),
        'arquivo_entrada':str(entrada), 'parquet_silver':str(destino)}
    (dirs['qualidade'] / 'relatorio_qualidade.json').write_text(json.dumps(relatorio, ensure_ascii=False, indent=2), encoding='utf-8')
    def sql_tipo(t):
        return 'BOOLEAN' if pa.types.is_boolean(t) else 'BIGINT' if pa.types.is_int64(t) else 'SMALLINT' if pa.types.is_integer(t) else 'DATE' if pa.types.is_date(t) else 'TEXT'
    ddl = '-- Apenas documentacao: este script nao executa SQL.\nCREATE SCHEMA IF NOT EXISTS silver;\nCREATE TABLE IF NOT EXISTS silver.populacao_ibge (\n'
    ddl += ',\n'.join(f'    "{f.name}" {sql_tipo(f.type)}' for f in esquema) + '\n);\n'
    (dirs['documentacao'] / 'criar_tabela_populacao_ibge.sql').write_text(ddl, encoding='utf-8')
    (dirs['documentacao'] / 'regras_tratamento.txt').write_text(
        'Grao: municipio de 7 digitos por ano_analise. Duplicatas preservadas e reportadas.\n'
        'Codigos 6/7 e UF conferidos entre si; nao equivalem a validacao contra cadastro territorial.\n'
        '2023 mantem referencia 2022 e cautela temporal. Fontes, datas e valor original preservados.\n'
        'Populacao positiva inteira ou NULL. Valores invalidos vao para quarentena.\n'
        'Populacao total nao e denominador especifico por idade/sexo. Agregar eventos antes do cruzamento.\n'
        'Usar apenas uma execucao Silver. Nao somar rodadas ou anos como populacao unica.\n'
        'VALIDADO_COM_RESSALVAS registra limites metodologicos, sem corrigir a fonte.\n', encoding='utf-8')
    print(resumo.to_string())
    print(f'\nStatus: {status}\nDados: {destino}\nQualidade: {dirs["qualidade"]}')
    return relatorio


if __name__ == '__main__':
    resultado_ibge_silver = main()

Lendo: D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil\Camada_Bronze\dados_ibge\populacao_municipios_2020_2024.parquet
             registros  populacoes_disponiveis
ano_analise                                   
2020              5571                    5570
2021              5571                    5570
2022              5570                    5570
2023              5570                    5570
2024              5571                    5570

Status: VALIDADO_COM_RESSALVAS
Dados: D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil\Camada_Silver\dados\populacao_ibge\20261024_163052_e1db3d2b\populacao_municipios_2020_2024.parquet
Qualidade: D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil\Camada_Silver\qualidade\populacao_ibge\20261024_163052_e1db3d2b


In [4]:
from pathlib import Path
import pandas as pd

arquivo = Path(
    r"D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil"
    r"\Camada_Silver\dados\populacao_ibge"
    r"\20261024_163052_e1db3d2b"
    r"\populacao_municipios_2020_2024.parquet"
)

populacao = pd.read_parquet(arquivo)

display(
    populacao.loc[
        populacao["populacao"].isna(),
        [
            "ano_analise", "codigo_ibge_7", "municipio",
            "uf", "valor_original", "status_populacao"
        ]
    ]
)

,ano_analise,codigo_ibge_7,municipio,uf,valor_original,status_populacao
5199,2020,5101837,Boa Esperança do Norte,MT,...,indisponivel_na_fonte
10770,2021,5101837,Boa Esperança do Norte,MT,...,indisponivel_na_fonte
27481,2024,5101837,Boa Esperança do Norte,MT,...,indisponivel_na_fonte


### Banco de dados PostgreSQL

In [1]:
from pathlib import Path
from getpass import getpass

import pyarrow.parquet as pq
import psycopg
from psycopg import sql

In [2]:
ARQUIVO = Path(
    r"D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil"
    r"\Camada_Silver\dados\populacao_ibge"
    r"\20261024_163052_e1db3d2b"
    r"\populacao_municipios_2020_2024.parquet"
)

def carregar_ibge():
    if not ARQUIVO.is_file():
        raise FileNotFoundError(f"Arquivo não encontrado: {ARQUIVO}")

    dados = pq.read_table(ARQUIVO)
    colunas = dados.column_names

    if dados.num_rows == 0:
        raise ValueError("O Parquet está vazio.")

    senha = getpass("Senha do PostgreSQL: ")

    try:
        conexao = psycopg.connect(
            host="localhost",
            port=5432,
            dbname="observatorio_covid19_brasil",
            user="postgres",
            password=senha,
            connect_timeout=15,
        )
    finally:
        del senha

    # Confirma toda a carga ao terminar; desfaz se ocorrer erro.
    with conexao:
        with conexao.cursor() as cursor:
            cursor.execute("SET LOCAL lock_timeout = '15s'")

            cursor.execute("""
                LOCK TABLE silver.populacao_ibge
                IN SHARE ROW EXCLUSIVE MODE
            """)

            cursor.execute("SELECT COUNT(*) FROM silver.populacao_ibge")
            existentes = cursor.fetchone()[0]

            if existentes:
                raise RuntimeError(
                    f"A tabela já contém {existentes:,} registros. "
                    "Carga cancelada para evitar duplicação."
                )

            cursor.execute("""
                SELECT column_name
                FROM information_schema.columns
                WHERE table_schema = 'silver'
                  AND table_name = 'populacao_ibge'
                ORDER BY ordinal_position
            """)
            colunas_banco = [linha[0] for linha in cursor.fetchall()]

            if set(colunas) != set(colunas_banco):
                raise ValueError(
                    "As colunas do Parquet e da tabela são diferentes.\n"
                    f"Somente no Parquet: {set(colunas) - set(colunas_banco)}\n"
                    f"Somente no banco: {set(colunas_banco) - set(colunas)}"
                )

            comando = sql.SQL(
                "COPY silver.populacao_ibge ({}) FROM STDIN"
            ).format(
                sql.SQL(", ").join(map(sql.Identifier, colunas))
            )

            print(f"Carregando {dados.num_rows:,} registros...", flush=True)

            with cursor.copy(comando) as copia:
                for lote in dados.to_batches(max_chunksize=5000):
                    for registro in lote.to_pylist():
                        copia.write_row(
                            tuple(registro[coluna] for coluna in colunas)
                        )

            cursor.execute("""
                SELECT
                    COUNT(*),
                    COUNT(*) FILTER (WHERE populacao IS NULL)
                FROM silver.populacao_ibge
            """)
            total, nulos = cursor.fetchone()

            if total != dados.num_rows:
                raise RuntimeError("Contagem divergente; carga será desfeita.")

            if nulos != dados.column("populacao").null_count:
                raise RuntimeError("Contagem de nulos divergente; carga será desfeita.")

            cursor.execute("""
                SELECT ano_analise, codigo_ibge_7
                FROM silver.populacao_ibge
                GROUP BY ano_analise, codigo_ibge_7
                HAVING COUNT(*) > 1
                LIMIT 1
            """)

            if cursor.fetchone() is not None:
                raise RuntimeError("Chave duplicada; carga será desfeita.")

    print("Carga concluída e confirmada!")
    print(f"Registros no banco: {total:,}")
    print(f"Registros sem população: {nulos}")


carregar_ibge()

Carregando 27,853 registros...
Carga concluída e confirmada!
Registros no banco: 27,853
Registros sem população: 3
